In [90]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

In [ ]:
FEATURE_NAMES = ("R", "W", "D", "S", "P", "O")

BOUNDS = {
    "R": (1400.0, 2000.0),
    "W": (40.0, 100.0),
    "D": (10.0, 30.0),
    "P": (0.03, 0.18),
}
CATEGORIES = {"S": ("square", "round"), "O": ("O1", "O2")}

# Hidden optimum per feature in [0, 1] (categoricals: 0 = first level, 1 = second).
_OPT = {"R": 0.65, "W": 0.35, "D": 0.55, "S": 1.0, "P": 0.45, "O": 0.0}


def make_feature_importance(active_features: list[str] | None = None) -> dict[str, float]:
    """Weight 1.0 for active features and 0.0 for the rest (None = all active)."""
    if active_features is None:
        return {name: 1.0 for name in FEATURE_NAMES}
    unknown = set(active_features) - set(FEATURE_NAMES)
    if unknown:
        raise ValueError(f"Unknown feature(s): {sorted(unknown)}")
    return {name: float(name in active_features) for name in FEATURE_NAMES}


def _normalize(name: str, value: Any) -> float:
    if name in CATEGORIES:
        if value not in CATEGORIES[name]:
            raise ValueError(f"{name} must be one of {CATEGORIES[name]}, got {value!r}")
        return float(CATEGORIES[name].index(value))
    lo, hi = BOUNDS[name]
    if not lo <= float(value) <= hi:
        raise ValueError(f"{name}={value} outside [{lo}, {hi}]")
    return (float(value) - lo) / (hi - lo)


def _sigmoid(x: float) -> float:
    return 1.0 / (1.0 + np.exp(-x))


def mock_welding_experiment(
    R: float,
    W: float,
    D: float,
    S: str,
    P: float,
    O: str,
    feature_importance: dict[str, float] | None = None,
    n_measurements: int = 1,
    sigma_p_noise_std: float = 8.0,
    uts_noise_std: float = 5.0,
    visual_noise_std: float = 0.4,
    seed: int | None = None,
) -> dict[str, Any]:
    """Mock welding response with controllable feature importance and noise.

    A feature with weight 0.0 has no effect on any output, e.g.
    feature_importance={"R": 1.0} makes outputs depend only on R.
    Returns n_measurements values each for sigma_p and uts, and one visual label.
    """
    if n_measurements < 1:
        raise ValueError("n_measurements must be >= 1")

    rng = np.random.default_rng(seed)
    raw = {"R": R, "W": W, "D": D, "S": S, "P": P, "O": O}
    weights = (
        make_feature_importance()
        if feature_importance is None
        else {n: float(feature_importance.get(n, 0.0)) for n in FEATURE_NAMES}
    )

    # Squared distance to the hidden optimum, per feature, weighted by importance.
    dist = {n: (_normalize(n, raw[n]) - _OPT[n]) ** 2 for n in FEATURE_NAMES}
    penalty = sum(weights[n] * dist[n] for n in FEATURE_NAMES)

    # Lower residual stress, higher UTS and better appearance near the optimum.
    sigma_p_mean = 120.0 + 300.0 * penalty
    uts_mean = 260.0 - 120.0 * penalty
    visual_score = 2.0 - 4.0 * penalty

    sigma_p = sigma_p_mean + rng.normal(0.0, sigma_p_noise_std, size=n_measurements)
    uts = uts_mean + rng.normal(0.0, uts_noise_std, size=n_measurements)
    visual = bool(rng.random() < _sigmoid(visual_score + rng.normal(0.0, visual_noise_std)))

    return {
        "sigma_peak": [float(x) for x in sigma_p],
        "uts": [float(x) for x in uts],
        "visual_quality": visual,
    }

In [92]:
mock_welding_experiment(
    R=1700, W=60, D=20, S="round", P=0.1, O="O1",
    feature_importance={"R": 1.0},
    n_measurements=3,
    seed=0,
)

{'sigma_peak': [127.75584176874715, 125.69316109366959, 131.87338120354624],
 'uts': [257.82450058576524, 254.62165313419445, 259.1079752745474],
 'visual_quality': True}

In [93]:
folder_path = Path(os.getcwd()) / "data"

In [94]:
exp = pd.read_excel(folder_path / "experiments.xlsx")
display(exp)

,trial_index,R,W,D,S,P,O,visual_quality,sigma_peak,uts
0,0,1700,70,20,round,0.105000,O2,1.0,127.755842,256.639476
1,1,1462,81,25,square,0.162470,O1,1.0,212.418007,228.246757
2,2,1733,44,18,round,0.102709,O2,0.0,124.219927,256.303258
3,3,1997,92,21,round,0.136314,O2,0.0,172.034853,232.938675
4,4,1649,63,13,square,0.055458,O1,0.0,131.353171,252.499414
5,5,1462,81,25,square,0.162470,O1,1.0,203.237882,217.516872
6,6,1687,73,19,round,0.079064,O2,1.0,137.265759,265.346123
7,7,1687,73,19,round,0.133737,O2,1.0,128.850675,257.957394
8,8,1435,70,14,round,0.104998,O2,1.0,211.114702,211.308453
9,9,1718,71,11,round,0.090116,O2,1.0,117.897305,259.486250


In [95]:
print(exp.columns)
cols_output = [
    "visual_quality",
    "sigma_peak",
    "uts",
]

Index(['trial_index', 'R', 'W', 'D', 'S', 'P', 'O', 'visual_quality',
       'sigma_peak', 'uts'],
      dtype='str')


In [96]:
filt_removed = exp[cols_output].isna().all(axis=1)
filt_removed

0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10     True
11     True
12     True
13     True
14     True
dtype: bool

In [97]:
exp_todo = exp[filt_removed].copy()
exp
display(exp)
display(exp_todo)

,trial_index,R,W,D,S,P,O,visual_quality,sigma_peak,uts
0,0,1700,70,20,round,0.105000,O2,1.0,127.755842,256.639476
1,1,1462,81,25,square,0.162470,O1,1.0,212.418007,228.246757
2,2,1733,44,18,round,0.102709,O2,0.0,124.219927,256.303258
3,3,1997,92,21,round,0.136314,O2,0.0,172.034853,232.938675
4,4,1649,63,13,square,0.055458,O1,0.0,131.353171,252.499414
5,5,1462,81,25,square,0.162470,O1,1.0,203.237882,217.516872
6,6,1687,73,19,round,0.079064,O2,1.0,137.265759,265.346123
7,7,1687,73,19,round,0.133737,O2,1.0,128.850675,257.957394
8,8,1435,70,14,round,0.104998,O2,1.0,211.114702,211.308453
9,9,1718,71,11,round,0.090116,O2,1.0,117.897305,259.486250


,trial_index,R,W,D,S,P,O,visual_quality,sigma_peak,uts
10,10,1462,81,25,square,0.162470,O1,NaN,NaN,NaN
11,11,1669,65,10,round,0.030000,O2,NaN,NaN,NaN
12,12,1746,82,10,square,0.141894,O1,NaN,NaN,NaN
13,13,1702,79,30,round,0.030000,O2,NaN,NaN,NaN
14,14,1766,86,10,square,0.030000,O1,NaN,NaN,NaN


In [ ]:
for idx, row in exp_todo.iterrows():
    result = mock_welding_experiment(
        R=row["R"],
        W=row["W"],
        D=row["D"],
        S=row["S"],
        P=row["P"],
        O=row["O"],
        feature_importance={"R": 1.0, "O": 1.0},
        n_measurements=1,
        seed=idx,
    )
    exp.loc[idx, "sigma_peak"] = result["sigma_peak"]
    exp.loc[idx, "visual_quality"] = float(result["visual_quality"])
    exp.loc[idx, "uts"] = result["uts"]


In [99]:
exp.to_excel(folder_path / "experiments.xlsx", index=False)